# 06. 매핑 + CLIP 기반 Top-5 향수 추천

1. 스타일 확률 × 매핑 행렬 → 5차원 향조 점수 → 향수별 향조 벡터와 코사인 유사도 (매핑 점수)
2. CLIP ViT-B/32로 패션 이미지와 `fragrance_text` 임베딩 간 코사인 유사도 (CLIP 점수)
3. `final = α·mapping + (1-α)·clip` (기본 α = 0.7) → Top-5


In [ ]:
!pip install git+https://github.com/openai/CLIP.git

In [ ]:
import pandas as pd
import numpy as np
import torch
import clip
from sklearn.metrics.pairwise import cosine_similarity
from PIL import Image

# fra_preprocessed.csv 로드
fra_df = pd.read_csv('fra_preprocessed.csv')
print(f"향수 데이터: {len(fra_df)}개")
print(fra_df[['perfume', 'brand', 'accord_sequence']].head(3))

향수 데이터: 24063개
                          perfume               brand  \
0  accento-overdose-pride-edition             xerjoff   
1            classique-pride-2024  jean-paul-gaultier   
2            classique-pride-2023  jean-paul-gaultier   

                                   accord_sequence  
0            rose, woody, fruity, aromatic, floral  
1        citrus, white floral, sweet, fresh, musky  
2  citrus, white floral, sweet, fresh spicy, musky  


In [ ]:
mapping_df = pd.read_csv('style_accord_mapping.csv', index_col=0)
print("mapping_df 로드 완료")
print(mapping_df)

mapping_df 로드 완료
            floral   intense   natural     niche  romantic
Romantic  0.350610  0.127928  0.157998  0.096680  0.266783
Casual    0.205964  0.131459  0.301761  0.184787  0.176030
Sporty    0.145126  0.096133  0.433129  0.204626  0.120986
Formal    0.149531  0.409816  0.106823  0.150316  0.183514
Street    0.120211  0.172242  0.192380  0.393613  0.121553
Vintage   0.156841  0.415732  0.101072  0.130639  0.195717


In [ ]:
# 모델 로드
clip_model, preprocess = clip.load("ViT-B/32")
clip_model.eval()

def build_text_embeddings(fra_df):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"사용 디바이스: {device}")

    texts = fra_df['fragrance_text'].fillna('').tolist()
    all_embeddings = []
    batch_size = 256

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        tokens = clip.tokenize(batch, truncate=True).to(device)
        with torch.no_grad():
            emb = clip_model.encode_text(tokens)
        all_embeddings.append(emb.cpu())
        print(f"  {min(i+batch_size, len(texts))}/{len(texts)} 완료")

    text_embeddings = torch.cat(all_embeddings, dim=0)
    torch.save(text_embeddings, 'text_embeddings.pt')
    print("저장 완료: text_embeddings.pt")
    return text_embeddings

# 실행
text_embeddings = build_text_embeddings(fra_df)
print(f"임베딩 shape: {text_embeddings.shape}")

def get_image_embedding(image_path):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    image = Image.open(image_path).convert('RGB')
    image_input = preprocess(image).unsqueeze(0).to(device)  # device 추가
    with torch.no_grad():
        embedding = clip_model.encode_image(image_input)
    return embedding.cpu().numpy()

def get_clip_scores(image_path, text_embeddings):
    """이미지-텍스트 코사인 유사도"""
    img_emb = get_image_embedding(image_path)
    text_emb = text_embeddings.numpy()
    scores = cosine_similarity(img_emb, text_emb)[0]
    return scores

사용 디바이스: cuda
저장 완료: text_embeddings.pt
임베딩 shape: torch.Size([24063, 512])


In [ ]:
# 실제 accord_sequence 값에 맞게 키워드 확장
ACCORD_MAPPING = {
    'floral':   ['floral', 'white floral', 'soft floral', 'rose', 'jasmine'],
    'intense':  ['intense', 'amber', 'oriental', 'warm spicy', 'spicy',
                 'cinnamon', 'oud'],
    'natural':  ['natural', 'fresh', 'citrus', 'aquatic', 'green',
                 'fresh spicy', 'aromatic'],
    'niche':    ['niche', 'woody', 'dry wood', 'earthy', 'smoky',
                 'vetiver', 'sandalwood'],
    'romantic': ['romantic', 'sweet', 'powdery', 'fruity', 'vanilla',
                 'musky', 'soapy'],
}
# def build_perfume_accord_vector(accord_sequence):
#     vec = np.zeros(5)
#     if not isinstance(accord_sequence, str):
#         return vec
#     for i, (category, keywords) in enumerate(ACCORD_MAPPING.items()):
#         if any(kw in accord_sequence.lower() for kw in keywords):
#             vec[i] = 1.0
#     return vec

# # 다시 계산
# fra_df['accord_vector'] = fra_df['accord_sequence'].apply(
#     build_perfume_accord_vector
# )

# # 확인
# print(fra_df[['accord_sequence', 'accord_vector']].head(5))
def build_perfume_accord_vector(accord_sequence):
    """
    단순 0/1 대신 등장 순서에 따라 가중치 부여
    앞에 나올수록 더 중요한 향조
    """
    vec = np.zeros(5)
    if not isinstance(accord_sequence, str):
        return vec

    accords = [a.strip() for a in accord_sequence.split(',')]
    total = len(accords)

    for rank, accord in enumerate(accords):
        weight = (total - rank) / total  # 앞에 있을수록 높은 가중치
        for i, (category, keywords) in enumerate(ACCORD_MAPPING.items()):
            if any(kw in accord.lower() for kw in keywords):
                vec[i] += weight

    # 정규화
    if vec.sum() > 0:
        vec = vec / vec.sum()

    return vec

# 다시 계산
fra_df['accord_vector'] = fra_df['accord_sequence'].apply(
    build_perfume_accord_vector
)


In [ ]:
def get_accord_scores(style_probs, mapping_df):
    """
    스타일 확률벡터 → 향조 점수 벡터

    예) style_probs = {'Casual': 0.6, 'Formal': 0.3, 'Street': 0.1}
    → 향조별 가중합 계산
    """
    accord_scores = np.zeros(len(ACCORD_COLS))

    for style, prob in style_probs.items():
        if style in mapping_df.index:
            accord_scores += prob * mapping_df.loc[style].values

    return accord_scores


def get_mapping_scores(style_probs, mapping_df, fra_df):
    """
    향조 점수 → 향수별 코사인 유사도 계산
    """
    # 쿼리 향조 벡터
    query_vec = get_accord_scores(style_probs, mapping_df).reshape(1, -1)

    # 향수 향조 행렬
    accord_matrix = np.array(fra_df['accord_vector'].tolist())

    # 코사인 유사도
    scores = cosine_similarity(query_vec, accord_matrix)[0]

    return scores

In [ ]:
def recommend_top5(style_probs, mapping_df, fra_df):
    """
    스타일 확률벡터 → Top-5 향수 추천
    """
    scores = get_mapping_scores(style_probs, mapping_df, fra_df)

    top5_idx = scores.argsort()[::-1][:5]
    top5 = fra_df.iloc[top5_idx][
        ['perfume', 'brand', 'accord_sequence']
    ].copy()
    top5['score'] = scores[top5_idx]

    return top5.reset_index(drop=True)

In [ ]:
def recommend_with_clip(style_probs, image_path,
                        mapping_df, fra_df,
                        text_embeddings, alpha=0.7):
    """
    매핑 점수 + CLIP 점수 결합한 최종 추천
    alpha: 매핑 점수 비중 (기본 0.7)
    """
    # 매핑 점수
    mapping_scores = get_mapping_scores(style_probs, mapping_df, fra_df)

    # CLIP 점수
    clip_scores = get_clip_scores(image_path, text_embeddings)

    # 정규화
    mapping_scores = mapping_scores / (mapping_scores.max() + 1e-9)
    clip_scores = clip_scores / (clip_scores.max() + 1e-9)

    # 결합
    final_scores = alpha * mapping_scores + (1 - alpha) * clip_scores

    top5_idx = final_scores.argsort()[::-1][:5]
    top5 = fra_df.iloc[top5_idx][
        ['perfume', 'brand', 'accord_sequence']
    ].copy()
    top5['final_score']   = final_scores[top5_idx]
    top5['mapping_score'] = mapping_scores[top5_idx]
    top5['clip_score']    = clip_scores[top5_idx]

    return top5.reset_index(drop=True)

In [ ]:
def recall_at_k(recommended_idx, relevant_set, k=3):
    top_k = set(recommended_idx[:k])
    hits = len(top_k & relevant_set)
    return hits / min(len(relevant_set), k)

def ndcg_at_k(recommended_idx, relevant_set, k=5):
    dcg = 0.0
    for i, idx in enumerate(recommended_idx[:k]):
        if idx in relevant_set:
            dcg += 1 / np.log2(i + 2)
    idcg = sum(1 / np.log2(i + 2) for i in range(min(len(relevant_set), k)))
    return dcg / idcg if idcg > 0 else 0.0

In [ ]:
ACCORD_COLS = ['floral', 'intense', 'natural', 'niche', 'romantic']
# 테스트용 스타일 확률벡터
style_probs = {
    'Casual': 0.6,
    'Formal': 0.3,
    'Street': 0.1
}

# mapping_df 로드 확인
print("=== mapping_df ===")
print(mapping_df)
print()

# 매핑 점수만 먼저 테스트 (CLIP 없이)
accord_scores = get_accord_scores(style_probs, mapping_df)
print("=== 향조 점수 ===")
for col, score in zip(mapping_df.columns, accord_scores):
    print(f"  {col}: {score:.4f}")
print()

# Top-5 추천 (매핑만)
result = recommend_top5(style_probs, mapping_df, fra_df)
print("=== Top-5 추천 결과 (매핑만) ===")
print(result)
print()

# ── 추가 1: 다른 스타일 조합 테스트 ──────────────────────
print("=== Romantic 위주 테스트 ===")
result2 = recommend_top5(
    {'Romantic': 0.7, 'Casual': 0.3},
    mapping_df, fra_df
)
print(result2)
print()

print("=== Vintage 위주 테스트 ===")
result3 = recommend_top5(
    {'Vintage': 0.8, 'Formal': 0.2},
    mapping_df, fra_df
)
print(result3)
print()

image_path = '11111.png'
print("=== CLIP 포함 Top-5 추천 ===")
result_clip = recommend_with_clip(
    style_probs, image_path,
    mapping_df, fra_df, text_embeddings,
    alpha=0.7
)
print(result_clip)
print()


image_path = '11111.png'
print("=== alpha 비율 실험 ===")
for alpha in [0.9, 0.7, 0.5]:
    result_a = recommend_with_clip(
        style_probs, image_path,
        mapping_df, fra_df, text_embeddings,
        alpha=alpha
    )
    print(f"alpha={alpha}")
    print(result_a[['perfume', 'final_score']])
    print()

=== mapping_df ===
            floral   intense   natural     niche  romantic
Romantic  0.350610  0.127928  0.157998  0.096680  0.266783
Casual    0.205964  0.131459  0.301761  0.184787  0.176030
Sporty    0.145126  0.096133  0.433129  0.204626  0.120986
Formal    0.149531  0.409816  0.106823  0.150316  0.183514
Street    0.120211  0.172242  0.192380  0.393613  0.121553
Vintage   0.156841  0.415732  0.101072  0.130639  0.195717

=== 향조 점수 ===
  floral: 0.1805
  intense: 0.2190
  natural: 0.2323
  niche: 0.1953
  romantic: 0.1728

=== Top-5 추천 결과 (매핑만) ===
                             perfume         brand  \
0                   secret-obsession  calvin-klein   
1      declaration-d-un-soir-intense       cartier   
2  soir-de-lune-edition-limitee-2015        sisley   
3                            montauk     bond-no-9   
4           new-york-gabriela-hearst  fueguia-1833   

                                     accord_sequence     score  
0  fresh spicy, woody, white floral, fruity, pow